# The Finite ONE — field game control

## 1 · Connect

Read-only. Confirms which account you are pointed at before anything mutates.

In [1]:
import time
from datetime import datetime, timedelta
from decimal import Decimal

import boto3
from boto3.dynamodb.conditions import Key

AWS_PROFILE = "kurohaka"
REGION      = "eu-west-3"

GAME_ID            = "finite-one"
STATE_TABLE        = "GameState"
AWARDS_TABLE       = "GameAwards"
PARTICIPANTS_TABLE = "Participants"
QUEUE_NAME         = "FiniteOneWorldPoints.fifo"

MAX_HEALTH = Decimal("100")

# team_0 is the staff team and does not play.
NOT_PLAYING = {"unassigned", "team_0", ""}

session = boto3.Session(profile_name=AWS_PROFILE, region_name=REGION)
ddb = session.resource("dynamodb")
sqs = session.client("sqs")

_id = session.client("sts").get_caller_identity()
print(f"profile : {AWS_PROFILE}")
print(f"region  : {REGION}")
print(f"account : {_id['Account']}")
print(f"identity: {_id['Arn']}")

profile : kurohaka
region  : eu-west-3
account : 369470881303
identity: arn:aws:iam::369470881303:user/kurohaka


#### METHODS

In [2]:
def now_ms():
    return int(time.time() * 1000)


def fetch_state(required=True):
    item = ddb.Table(STATE_TABLE).get_item(Key={"game_id": GAME_ID}).get("Item")
    if not item and required:
        raise RuntimeError("No game found in GameState. Run the Start cell first.")
    return item


def projected_health(state, now=None):
    """Mirrors game_state.project_health, so this matches what players see."""
    now = now or now_ms()
    health = Decimal(str(state.get("world_health", MAX_HEALTH)))
    if state.get("status") != "running":
        return max(Decimal(0), min(MAX_HEALTH, health))
    last_tick = int(state.get("last_tick_ms") or 0)
    if not last_tick or now <= last_tick:
        return max(Decimal(0), min(MAX_HEALTH, health))
    pace = Decimal(str(state.get("pace", 1)))
    elapsed = Decimal(now - last_tick) / Decimal(60000)
    return max(Decimal(0), min(MAX_HEALTH, health - pace * elapsed))


def time_left(health, pace):
    """Minutes to collapse assuming NO team adds any HP to the world.

    Returns (minutes, human text). Sacrifices push this out; nothing else does.
    """
    health, pace = Decimal(str(health)), Decimal(str(pace))
    if pace <= 0:
        return None, "never - the meter is frozen (pace 0)"
    if health <= 0:
        return 0.0, "already collapsed"
    minutes = float(health / pace)
    hours, mins = divmod(int(round(minutes)), 60)
    span = f"{hours}h {mins:02d}m" if hours else f"{mins}m"
    eta = datetime.now().astimezone() + timedelta(minutes=minutes)
    return minutes, f"{span}  (~{eta:%H:%M %Z}"


def show_status(state=None):
    state = state or fetch_state()
    now = now_ms()
    health = projected_health(state, now)
    pace = Decimal(str(state.get("pace", 1)))
    status = state.get("status", "idle")

    print(f"  status       : {status}")
    print(f"  world health : {health:.1f} / 100")
    print(f"  pace         : {pace} HP per minute")

    started = int(state.get("started_at_ms") or 0)
    if started:
        print(f"  elapsed      : {(now - started) / 60000:.0f} min")

    if status == "running":
        _, text = time_left(health, pace)
        print(f"  time left    : {text}")
    else:
        print("  time left    : n/a - decay only runs while status is 'running'")

    scores = state.get("scores") or {}
    print(f"  teams        : {len(scores)}")
    if scores:
        top = sorted(scores.items(), key=lambda kv: (-int(kv[1]), int(kv[0])))[:3]
        print("  leading      : " + ", ".join(f"team {t} ({int(p)})" for t, p in top))
    return state

def playing_teams():
    """The teams on the roster - the only place that knows who plays."""
    table = ddb.Table(PARTICIPANTS_TABLE)
    teams, kwargs = set(), {"ProjectionExpression": "team_id"}
    while True:
        res = table.scan(**kwargs)
        for item in res.get("Items", []):
            team_id = item.get("team_id") or ""
            if team_id in NOT_PLAYING:
                continue
            number = "".join(c for c in team_id if c.isdigit())
            if number:
                teams.add(str(int(number)))
        if "LastEvaluatedKey" in res:
            kwargs["ExclusiveStartKey"] = res["LastEvaluatedKey"]
        else:
            break
    if not teams:
        raise RuntimeError("No teams on the roster. Seed Participants first.")
    return sorted(teams, key=int)


def start_game(health=100.0, pace=1.0, force=False):
    if not 0 < health <= 100:
        raise ValueError("health must be in (0, 100]")
    if pace < 0:
        raise ValueError("pace cannot be negative; use 0 to freeze the meter")

    current = fetch_state(required=False)
    if current and current.get("status") == "running" and not force:
        raise RuntimeError(
            "A game is already running - starting again would zero every score. "
            "Pass force=True if that is what you want."
        )

    teams = playing_teams()
    now = now_ms()
    ddb.Table(STATE_TABLE).put_item(Item={
        "game_id": GAME_ID,
        "status": "running",
        "world_health": Decimal(str(health)),
        "pace": Decimal(str(pace)),
        "last_tick_ms": now,
        "started_at_ms": now,
        "scores": {team: 0 for team in teams},
        "version": 0,
    })
    print(f"Started with {len(teams)} teams, all at 0 points.\n")

def clear_ledger():
    table = ddb.Table(AWARDS_TABLE)
    deleted = 0
    kwargs = {
        "IndexName": "byGame",
        "KeyConditionExpression": Key("game_id").eq(GAME_ID),
        "ProjectionExpression": "award_id",
    }
    while True:
        res = table.query(**kwargs)
        with table.batch_writer() as batch:
            for item in res.get("Items", []):
                batch.delete_item(Key={"award_id": item["award_id"]})
                deleted += 1
        if "LastEvaluatedKey" in res:
            kwargs["ExclusiveStartKey"] = res["LastEvaluatedKey"]
        else:
            break
    print(f"  - {deleted} ledger entries deleted")


def purge_queue():
    try:
        url = sqs.get_queue_url(QueueName=QUEUE_NAME)["QueueUrl"]
    except Exception as err:
        print(f"  ! could not resolve the world-points queue: {err}")
        return
    try:
        sqs.purge_queue(QueueUrl=url)
        print("  - world-points queue purged")
    except sqs.exceptions.PurgeQueueInProgress:
        print("  - queue purge already in progress (allowed once per 60s), skipped")


def reset_game(pace=1.0):
    """Clean slate: no awards, no queued sacrifices, all scores 0, game idle."""
    print("Resetting:")
    clear_ledger()
    purge_queue()

    teams = playing_teams()
    ddb.Table(STATE_TABLE).put_item(Item={
        "game_id": GAME_ID,
        # idle, not running - tick.py no-ops until you run the Start cell.
        "status": "idle",
        "world_health": MAX_HEALTH,
        "pace": Decimal(str(pace)),
        "last_tick_ms": 0,
        "scores": {team: 0 for team in teams},
        "version": 0,
    })
    print(f"  - {len(teams)} teams reseeded at 0 points\n")
    print("Reset complete. The game is idle - run the Start cell to begin.\n")

def set_pace(pace):
    """Change the decay speed of a game that exists. Safe mid-game."""
    if pace < 0:
        raise ValueError("pace cannot be negative; use 0 to freeze the meter")

    state = fetch_state()          # a game must exist to re-pace it
    now = now_ms()
    settled = projected_health(state, now)   # charge the old pace first
    old = Decimal(str(state.get("pace", 1)))

    ddb.Table(STATE_TABLE).update_item(
        Key={"game_id": GAME_ID},
        UpdateExpression=(
            "SET world_health = :h, last_tick_ms = :now, #pace = :pace, "
            "#v = #v + :one"
        ),
        ExpressionAttributeNames={"#pace": "pace", "#v": "version"},
        ExpressionAttributeValues={
            ":h": settled,
            ":now": now,
            ":pace": Decimal(str(pace)),
            ":one": 1,
        },
    )
    print(f"Pace {old} -> {pace} HP/min (settled at {settled:.1f} HP)\n")


def pace_table(paces=(0.5, 1.0, 1.5, 2.0, 3.0), health=None):
    """Time left at each pace, assuming NO team sacrifices anything.

    Writes nothing, and works before the game exists - which is exactly when
    you want it, to pick a starting pace.
    """
    if health is None:
        state = fetch_state(required=False)
        if state:
            health = projected_health(state)
            print(f"live game: {health:.1f} HP, status '{state.get('status', 'idle')}'\n")
        else:
            health = MAX_HEALTH
            print(f"no game yet - assuming a fresh start at {health:.0f} HP\n")
    health = Decimal(str(health))

    for pace in paces:
        _, text = time_left(health, pace)
        print(f"  pace {pace:>5}min  ->  {text}")

def end_game():
    ddb.Table(STATE_TABLE).update_item(
        Key={'game_id': GAME_ID},
        UpdateExpression='SET #s = :ended, ended_at_ms = :now',
        ExpressionAttributeNames={'#s': 'status'},
        ExpressionAttributeValues={':ended': 'ended', ':now': now_ms()},
    )
    print('Game ended. The meter is frozen where it stands.')

## 2 · Status

Read-only, safe to re-run at any time. `time left` is the answer to *how long until the world dies if no team sacrifices anything*.

In [28]:
show_status()

  status       : running
  world health : 8.7 / 100
  pace         : 1 HP per minute
  elapsed      : 91 min
  time left    : 9m  (~11:34 CEST, if no team sacrifices anything)
  teams        : 30
  leading      : team 1 (0), team 2 (0), team 3 (0)


{'game_id': 'finite-one',
 'last_tick_ms': Decimal('1787822725158'),
 'pace': Decimal('1'),
 'scores': {'22': Decimal('0'),
  '18': Decimal('0'),
  '10': Decimal('0'),
  '13': Decimal('0'),
  '17': Decimal('0'),
  '27': Decimal('0'),
  '16': Decimal('0'),
  '5': Decimal('0'),
  '9': Decimal('0'),
  '26': Decimal('0'),
  '23': Decimal('0'),
  '7': Decimal('0'),
  '11': Decimal('0'),
  '19': Decimal('0'),
  '25': Decimal('0'),
  '30': Decimal('0'),
  '6': Decimal('0'),
  '28': Decimal('0'),
  '29': Decimal('0'),
  '3': Decimal('0'),
  '8': Decimal('0'),
  '15': Decimal('0'),
  '2': Decimal('0'),
  '4': Decimal('0'),
  '21': Decimal('0'),
  '20': Decimal('0'),
  '24': Decimal('0'),
  '1': Decimal('0'),
  '12': Decimal('0'),
  '14': Decimal('0')},
 'started_at_ms': Decimal('1787817297221'),
 'status': 'running',
 'version': Decimal('93'),
 'world_health': Decimal('9.534383333333333333333333327')}

## 3 · Start

Writes a fresh state item, so **every score returns to 0** and the team list is re-read from the roster. Refuses to clobber a running game unless `force=True`.

In [4]:
START_HEALTH = 100.0
START_PACE   = 0.5

start_game(health=START_HEALTH, pace=START_PACE)

Started with 30 teams, all at 0 points.



## 4 · Reset

Wipes the award ledger and purges the world-points queue, then parks the game at `idle` with every score zeroed, so a rehearsal cannot leak sacrifices into the real run. Requires typing the confirmation token.

In [3]:
reset_game()

Resetting:
  - 4 ledger entries deleted
  - world-points queue purged
  - 30 teams reseeded at 0 points

Reset complete. The game is idle - run the Start cell to begin.



## 5 · Pace

`pace` is HP lost per minute. It is private: it lives only in DynamoDB and never
reaches an API response or the client bundle.

Safe to change mid-game. The cell settles the minutes already owed **at the old
pace** before the new one applies, so the meter bends rather than jumping — a
change never retroactively rewrites time the world already survived.

`pace = 0` freezes the meter.

In [6]:
# ---- compare before committing (read-only) ---------------------------------
pace_table(paces=(0.5, 1.0, 1.5, 2.0, 3.0))

# ---- uncomment to apply (needs a started game) -----------------------------
set_pace(20)

live game: 97.1 HP, status 'running'

  pace   0.5min  ->  3h 14m  (~22:58 CEST
  pace   1.0min  ->  1h 37m  (~21:20 CEST
  pace   1.5min  ->  1h 05m  (~20:48 CEST
  pace   2.0min  ->  49m  (~20:32 CEST
  pace   3.0min  ->  32m  (~20:16 CEST
Pace 1 -> 20 HP/min (settled at 97.1 HP)

